In [ ]:
!pip install -q transformers accelerate librosa soundfile jiwer datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 19.7 MB/s eta 0:00:00


In [ ]:
import json

with open("/content/quran_data.json", "r", encoding="utf-8") as f:
    quran_data = json.load(f)

print(type(quran_data))
print(len(quran_data))
print(quran_data[0])

<class 'list'>
114
{'id': 1, 'name': 'الفاتحة', 'transliteration': 'Al-Fatihah', 'type': 'meccan', 'total_verses': 7, 'verses': [{'id': 1, 'text': 'بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ'}, {'id': 2, 'text': 'ٱلۡحَمۡدُ لِلَّهِ رَبِّ ٱلۡعَٰلَمِينَ'}, {'id': 3, 'text': 'ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ'}, {'id': 4, 'text': 'مَٰلِكِ يَوۡمِ ٱلدِّينِ'}, {'id': 5, 'text': 'إِيَّاكَ نَعۡبُدُ وَإِيَّاكَ نَسۡتَعِينُ'}, {'id': 6, 'text': 'ٱهۡدِنَا ٱلصِّرَٰطَ ٱلۡمُسۡتَقِيمَ'}, {'id': 7, 'text': 'صِرَٰطَ ٱلَّذِينَ أَنۡعَمۡتَ عَلَيۡهِمۡ غَيۡرِ ٱلۡمَغۡضُوبِ عَلَيۡهِمۡ وَلَا ٱلضَّآلِّينَ'}]}


In [ ]:
!wget -c "https://everyayah.com/data/Abdul_Basit_Murattal_192kbps/000_versebyverse.zip" \
    -O /content/audio.zip

--2026-08-25 00:57:17--  https://everyayah.com/data/Abdul_Basit_Murattal_192kbps/000_versebyverse.zip
Resolving everyayah.com (everyayah.com)... 138.199.36.9, 2400:52e0:1e00::1054:1
Connecting to everyayah.com (everyayah.com)|138.199.36.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2703478666 (2.5G) [application/zip]
Saving to: ‘/content/audio.zip’

/content/audio.zip  100%[===================>]   2.52G  35.9MB/s    in 77s     

2026-08-25 00:58:34 (33.7 MB/s) - ‘/content/audio.zip’ saved [2703478666/2703478666]



In [ ]:
!unzip -q audio.zip -d audio

In [ ]:
import os

audio_path = "/content/audio"

print("Number of files:", len(os.listdir(audio_path)))
print("First 10 files:")
print(os.listdir(audio_path)[:10])

Number of files: 6353
First 10 files:
['003181.mp3', '043078.mp3', '040012.mp3', '068011.mp3', '028054.mp3', '071008.mp3', '008012.mp3', '030010.mp3', '012037.mp3', '037022.mp3']


In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
Tesla T4


In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
import torch

model_id = "naazimsnh02/whisper-large-v3-turbo-ar-quran"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

model = model.to("cuda")

print("Model loaded successfully!")
print("Device:", model.device)

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0


In [ ]:
no_timestamps_id = processor.tokenizer.convert_tokens_to_ids("<|notimestamps|>")

model.generation_config.no_timestamps_token_id = no_timestamps_id

print("no_timestamps_token_id:", no_timestamps_id)

no_timestamps_token_id: 50364


In [ ]:
from transformers import pipeline
import torch

pipe = pipeline(
    "automatic-speech-recognition",
    model="naazimsnh02/whisper-large-v3-turbo-ar-quran",
    device=0,
    torch_dtype=torch.float16,
    chunk_length_s=30,
    stride_length_s=5
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Model loaded successfully!


In [ ]:
from transformers import pipeline
import torch

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0,
    torch_dtype=torch.float16,
    chunk_length_s=30,
    stride_length_s=5
)

print("Pipeline ready!")

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Pipeline ready!


In [ ]:
import os
import json

# Audio file
audio_file = "/content/audio/001001.mp3"

# Get Surah and Ayah number from filename
filename = os.path.basename(audio_file).replace(".mp3", "")
surah_number = int(filename[:3])
ayah_number = int(filename[3:])

# Get reference text from Quran dataset
surah = quran_data[surah_number - 1]
ayah = surah["verses"][ayah_number - 1]

print("Surah:", surah["name"])
print("Ayah:", ayah_number)
print("Reference:")
print(ayah["text"])

Surah: الفاتحة
Ayah: 1
Reference:
بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ


In [ ]:
import zipfile
import os

zip_path = "/content/audio.zip"
extract_path = "/content/audio"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Audio extracted successfully!")

Audio extracted successfully!


In [ ]:
audio_files = []

for root, dirs, files in os.walk("/content/audio"):
    for file in files:
        if file.endswith(".mp3"):
            audio_files.append(os.path.join(root, file))

print("Number of audio files:", len(audio_files))
print("First 10 files:")

for file in audio_files[:10]:
    print(os.path.basename(file))

Number of audio files: 6350
First 10 files:
003181.mp3
043078.mp3
040012.mp3
068011.mp3
028054.mp3
071008.mp3
008012.mp3
030010.mp3
012037.mp3
037022.mp3


In [ ]:
import re

def normalize_arabic(text):
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)
    text = text.replace('ٱ', 'ا')
    text = text.replace('إ', 'ا')
    text = text.replace('أ', 'ا')
    text = text.replace('آ', 'ا')
    text = text.replace('ى', 'ي')
    return text.strip()

In [ ]:
from transformers import pipeline
import torch

pipe = pipeline(
    "automatic-speech-recognition",
    model="naazimsnh02/whisper-large-v3-turbo-ar-quran",
    device=0,
    torch_dtype=torch.float16
)

result = pipe("/content/audio/001001.mp3")

print("Predicted:")
print(result["text"])

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Predicted:
<|startoftranscript|><|ar|><|transcribe|><|notimestamps|>بِسْمِ اللَّهِ الرَّحْمَنِ الرَّحِيمِ


In [ ]:
audio_path = "/content/audio/002286.mp3"

result = pipe(audio_path)

print("Whisper Transcription:")
print(result["text"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Whisper Transcription:
لَا يُكَلِّفُ اللَّهُ نَفْسًا إِلَّا وُسْعَهَا لَهَا مَا كَسَبَتْ وَعَلَيْهَا مَا اكْتَسَبَتْ رَبَّنَا لَا تُؤَاخِذْنَا إِنْ نَسِينَا أَوْ أَخْطَأْنَا رَبَّنَا وَلَا تَحْمِلْ عَلَيْنَا إِصْرًا كَمَا حَمَلْتَهُ عَلَى الَّذِينَ مِنْ قَبْلِنَا رَبَّنَا وَلَا تُحَمِّلْنَا مَا لَا طَاقَةَ لَنَا بِهِ وَاعْفُ عَنَّا وَاغْفِرْ لَنَا وَارْحَمْنَا أَنْتَ مَوْلَانَا فَانْصُرْنَا عَلَى قَوْمِ الْكَافِرِينَ


In [ ]:
audio_path = "/content/audio/112001.mp3"

result = pipe(audio_path)

print("Whisper Transcription:")
print(result["text"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Whisper Transcription:
قُلْ هُوَ اللَّهُ أَحَدٌ


In [ ]:
audio_path = "/content/audio/002255.mp3"

result = pipe(audio_path)

print("Whisper Transcription:")
print(result["text"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Whisper Transcription:
اللَّهُ لَا إِلَهَ إِلَّا هُوَ الْحَيُّ الْقَيُّومُ لَا تَأْخُذُهُ سِنَةٌ وَلَا نَوْمٌ لَهُ مَا فِي السَّمَاوَاتِ وَمَا فِي الْأَرْضِ مَنْ ذَا الَّذِي يَشْفَعُ عِنْدَهُ إِلَّا بِإِذْنِهِ يَعْلَمُ مَا بَيْنَ أَيْدِيهِمْ وَمَا خَلْفَهُمْ وَلَا يُحِيطُونَ بِشَيْءٍ مِنْ عِلْمِهِ إِلَّا بِمَا شَاءُ وَسِعَ كُرْسِيُّهُ السَّمَاوَاتِ وَالْأَرْضَ وَلَا يَؤُودُهُ حِفْظُهُمَا وَهُوَ الْعَلِيُّ الْعَظِيمُ


In [ ]:
from google.colab import output
from IPython.display import display, HTML
import base64

def save_audio(data):
    audio_bytes = base64.b64decode(data.split(",")[1])

    with open("/content/my_recording.webm", "wb") as f:
        f.write(audio_bytes)

    print("Recording saved successfully!")

output.register_callback("notebook.save_audio", save_audio)

display(HTML("""
<div style="text-align:center; margin:20px;">

    <button id="startBtn"
            style="font-size:18px; padding:12px 25px; margin:5px;">
        🎙️ Start Recording
    </button>

    <button id="stopBtn"
            disabled
            style="font-size:18px; padding:12px 25px; margin:5px;">
        ⏹️ Stop & Send
    </button>

    <p id="status">Ready</p>
</div>

<script>
let recorder = null;
let chunks = [];
let stream = null;

document.getElementById("startBtn").onclick = async function() {

    try {
        stream = await navigator.mediaDevices.getUserMedia({
            audio: true
        });

        recorder = new MediaRecorder(stream);
        chunks = [];

        recorder.ondataavailable = function(event) {
            if (event.data.size > 0) {
                chunks.push(event.data);
            }
        };

        recorder.onstop = function() {

            const blob = new Blob(chunks, {
                type: "audio/webm"
            });

            const reader = new FileReader();

            reader.onloadend = function() {

                google.colab.kernel.invokeFunction(
                    "notebook.save_audio",
                    [reader.result],
                    {}
                );
            };

            reader.readAsDataURL(blob);

            stream.getTracks().forEach(
                track => track.stop()
            );
        };

        recorder.start();

        document.getElementById("startBtn").disabled = true;
        document.getElementById("stopBtn").disabled = false;
        document.getElementById("status").innerText =
            "🔴 Recording... Speak now!";

    } catch (error) {

        document.getElementById("status").innerText =
            "❌ Microphone permission denied";

        console.error(error);
    }
};


document.getElementById("stopBtn").onclick = function() {

    if (recorder && recorder.state !== "inactive") {

        recorder.stop();

        document.getElementById("startBtn").disabled = false;
        document.getElementById("stopBtn").disabled = true;

        document.getElementById("status").innerText =
            "⏳ Saving recording...";
    }
};
</script>
"""))

Recording saved successfully!


In [ ]:
from google.colab import output
import base64

def save_recording(data):
    audio_data = base64.b64decode(data)

    with open("/content/my_recording.webm", "wb") as f:
        f.write(audio_data)

    print("Recording saved successfully!")

output.register_callback(
    "notebook.save_recording",
    save_recording
)

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<div>
    <button id="startBtn" onclick="startRecording()">
        🎙️ Start Recording
    </button>

    <button id="stopBtn" onclick="stopRecording()" disabled>
        ⏹️ Stop & Send
    </button>

    <p id="status">Ready</p>
</div>

<script>
let recorder;
let chunks = [];

async function startRecording() {
    const stream = await navigator.mediaDevices.getUserMedia({audio: true});

    recorder = new MediaRecorder(stream);
    chunks = [];

    recorder.ondataavailable = event => {
        chunks.push(event.data);
    };

    recorder.onstop = async () => {
        const blob = new Blob(chunks, {type: 'audio/webm'});
        const reader = new FileReader();

        reader.onloadend = () => {
            const base64data = reader.result.split(',')[1];

            google.colab.kernel.invokeFunction(
                'notebook.save_recording',
                [base64data],
                {}
            );
        };

        reader.readAsDataURL(blob);

        stream.getTracks().forEach(track => track.stop());
    };

    recorder.start();

    document.getElementById("startBtn").disabled = true;
    document.getElementById("stopBtn").disabled = false;
    document.getElementById("status").innerText = "🔴 Recording...";
}

function stopRecording() {
    recorder.stop();

    document.getElementById("startBtn").disabled = false;
    document.getElementById("stopBtn").disabled = true;
    document.getElementById("status").innerText = "Processing...";
}
</script>
"""))

In [ ]:
import subprocess

input_audio = "/content/my_recording.webm"
output_audio = "/content/my_recording.wav"

subprocess.run([
    "ffmpeg",
    "-y",
    "-i", input_audio,
    "-ar", "16000",
    "-ac", "1",
    output_audio
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print("Audio converted successfully!")

Audio converted successfully!


In [ ]:
result = pipe("/content/my_recording.webm")

print("Your Recitation:")
print(result["text"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Your Recitation:
 بِسْمِ اللَّهِ الرَّحْمَنِ الرَّحِيمِ
